# PWV03c_longscale : Two-Point Temporal Correlation — long timescales via pyzdcf

## Overview

This notebook is the long-timescale counterpart of
`PWV03c_TwoPoint_TemporalCorrelation_separateFilters_pyzdcf.ipynb`.
It uses the **pyzdcf** library to compute the Discrete Covariance Function
(DCF) of the PWV time series at Cerro Pachon on **multi-day timescales**
(hours to ~500 days).

## Key difference from PWV03c (intra-night)

In the intra-night version a nightly mean is subtracted and only same-night
pairs are analysed (lags up to 10 h).  Here:

1. **No nightly mean subtraction**: the raw (quality-cut) PWV values are
   passed directly to pyzdcf after a simple global mean removal through
   pyzdcf's internal normalisation.  This preserves the inter-night PWV
   modulation driven by synoptic weather and the seasonal cycle.

2. **All pairs across the full dataset are included**: pyzdcf operates on the
   complete time series without any night-boundary constraint, so lags range
   from the minimum intra-night cadence to the full multi-year span.

3. Two variants are run:
   - **All filters combined**: gives the maximum statistical power.
   - **Per-filter**: allows cross-checking that the decorrelation is
     filter-independent (as expected for a purely atmospheric quantity).

## pyzdcf library

pyzdcf implements the Z-transformed Discrete Correlation Function of
Alexander (1997), which is designed for unevenly sampled time series.  It
bins pairs by equal pair counts (not equal lag intervals) and applies a
Fisher z-transform before averaging, which avoids the bias of the standard
DCF at small-number statistics.  Monte Carlo error estimation is included.

Input format: a 3-column CSV `(time, value, sigma)` with no header.

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-29  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pyzdcf import pyzdcf

%matplotlib inline

mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# Output directory for figures
pathfigs = 'figs_PWV03longscale'
prefix   = 'pwv03c_longscale_zdcf'
os.makedirs(pathfigs, exist_ok=True)
figtype  = '.pdf'

# pyzdcf I/O directories (reuse existing data_pyzdcf)
pathdata_zdcf   = 'data_pyzdcf'
dcf_path_input  = os.path.join(pathdata_zdcf, 'dcf_timecurves_longscale')
dcf_path_output = os.path.join(pathdata_zdcf, 'dcf_results_longscale')
for d in [pathdata_zdcf, dcf_path_input, dcf_path_output]:
    os.makedirs(d, exist_ok=True)

In [ ]:
FLAG_WITHCOLLIMATOR     = True
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')

# Instrumental repeatability (mm) — used as measurement sigma for pyzdcf
SIGMA_REPEATABILITY = 0.12

---
## 1. Load data & build Time column

In [ ]:
import sys
sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split('/')[-1]
print(f'Loading: {atmfilename}')

if 'parquet' in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

df_spec['Time'] = pd.to_datetime(df_spec['DATE-OBS'], utc=True)
df_spec = df_spec.sort_values('Time', ascending=True).reset_index(drop=True)

if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['Time'] > datetime_WITHCOLLIMATOR].copy()

print(f'Shape: {df_spec.shape}')
print(f'Date range: {df_spec["Time"].min().date()}  ->  {df_spec["Time"].max().date()}')

In [ ]:
df_spec['nightObs'] = df_spec.apply(lambda x: x['id'] // 100_000, axis=1)
df_spec['seq_num']  = df_spec['id'] % 100_000
df_spec['night_from_time_utc'] = df_spec['Time'].dt.strftime('%Y%m%d').astype(int)

FLAG_CORRECT_NIGHTOBS_VARIABLES = True
if FLAG_CORRECT_NIGHTOBS_VARIABLES and 'run2026_v02' in version_run:
    print('COMPUTE NIGHTOBS FROM Time')
    df_spec['Time_local'] = df_spec['Time'].dt.tz_convert('America/Santiago')
    df_spec['nightObs_corr'] = (
        (df_spec['Time_local'] - pd.Timedelta(hours=12))
        .dt.strftime('%Y%m%d')
        .astype(int)
    )
    df_spec['nightObs'] = df_spec['nightObs_corr']

In [ ]:
if 'run2026_v02' in version_run:
    df_spec['chi2_ram'] = df_spec['CHI2_FIT']
    df_spec['chi2_rum'] = df_spec['CHI2_FIT']

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec['FILTER'].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ['PWV [mm]_ram', 'PWV [mm]_rum', 'PWV [mm]_err_ram', 'PWV [mm]_err_rum']
df_spec.dropna(subset=pwv_cols, inplace=True)
print(f'After filter selection: {len(df_spec)} rows')

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='CHI2_FIT', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_ram', ext='norm')
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col='TARGET', filter_col='FILTER',
    feature_col='chi2_rum', ext='norm')

---
## 2. Quality cuts

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

pathdata = 'data_PWV01seas'
if FLAG_LOOSE_CUTS:
    filename_cuts = f'{pathdata}/cuts_loose_finaldecision.json'
    cutstype_name = 'loose-cuts'
elif FLAG_TIGHT_CUTS:
    filename_cuts = f'{pathdata}/cuts_tight_finaldecision.json'
    cutstype_name = 'tight-cuts'
else:
    filename_cuts = f'{pathdata}/cuts_finaldecision.json'
    cutstype_name = 'standard-cuts'

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col='id')
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on='id')
df_keep     = df_selected[df_selected['pass_all_cuts']].copy()

print(f'{cutstype_name}: {len(df_keep)} / {len(df_spec)} kept')

PLOT_XSCALE = "linear"
PLOT_YMIN = -0.5
PLOT_YMAX = 1.2

---
## 3. pyzdcf helper functions

In [ ]:
def ComputeZDCF(filename_in, df_pwv_curve, minpts=10, num_MC=200):
    """
    Compute the Z-transformed Discrete Correlation Function with pyzdcf.

    Parameters
    ----------
    filename_in   : str  — name of the temporary CSV written to dcf_path_input.
    df_pwv_curve  : pd.DataFrame — 3 columns: (time_days, pwv_mm, sigma_mm), no header.
    minpts        : int  — minimum number of pairs per bin (0 = pyzdcf default = 11).
    num_MC        : int  — number of Monte Carlo iterations for error estimation.

    Returns
    -------
    dcf_df : pd.DataFrame — pyzdcf output (columns: tau, dcf, -err, +err, ...).
    """
    full_filename_in = os.path.join(dcf_path_input, filename_in)
    df_pwv_curve.to_csv(full_filename_in, index=False, header=False)

    params_dcf = dict(
        autocf           = True,
        prefix           = 'acf',
        uniform_sampling = False,
        omit_zero_lags   = False,
        minpts           = minpts,
        num_MC           = num_MC,
        lc1_name         = filename_in,
        lc2_name         = filename_in,
    )

    dcf_df = pyzdcf(
        input_dir  = dcf_path_input  + '/',
        output_dir = dcf_path_output + '/',
        intr       = False,
        parameters = params_dcf,
        sep        = ',',
        sparse     = 'auto',
        verbose    = False,
    )
    return dcf_df


def plot_zdcf(ax, dcf_df, label='', color='steelblue',
              xscale='linear', xlim=None, ylim=(-1.2, 1.2)):
    """Plot a pyzdcf output on the given Axes."""
    x    = dcf_df['tau'].values
    y    = dcf_df['dcf'].values
    xerr = dcf_df[["-sig(tau)", "+sig(tau)"]].values.T
    yerr = dcf_df[["-err(dcf)", "+err(dcf)"]].values.T

    ax.errorbar(
        x, y, xerr=xerr, yerr=yerr,
        fmt='o', color=color, ms=3, lw=0.8,
        ecolor=color, elinewidth=1.5, capsize=2,
        label=label
    )
    ax.axhline(0.0,      ls='--', color='gray',       lw=1.0)
    ax.axhline(1.0/np.e, ls=':',  color='darkorange', lw=1.5, label=r'$C=1/e$')
    ax.set_xscale(xscale)
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_ylabel('ZDCF', fontsize=13)
    ax.legend(fontsize=9, loc='upper right')

---
## 4. All-filter ZDCF — full dataset, time in days

pyzdcf uses the raw PWV values (not mean-subtracted): it internally
standardises the series.  The sigma column is set to the instrumental
repeatability $\sigma_{\rm rep}$ as a constant floor.

In [ ]:
# ── Build pyzdcf input: (time_days, pwv_mm, sigma_mm) ─────────────────────────
df_keep['PWV']   = df_keep['PWV [mm]_ram']
t0_mjd           = df_keep['MJD'].min()
df_keep['t_day'] = df_keep['MJD'] - t0_mjd

df_dcf_all = df_keep[['t_day', 'PWV']].copy()
df_dcf_all['sigma'] = SIGMA_REPEATABILITY

print(f'pyzdcf input: {len(df_dcf_all)} rows')
print(f'Time span: {df_dcf_all["t_day"].max():.1f} days')
df_dcf_all.head()

In [ ]:
# ── Run pyzdcf (minpts=21 for good statistics per bin) ────────────────────────
print('Running pyzdcf on all filters...')
dcf_all = ComputeZDCF('zdcf_longscale_all.csv', df_dcf_all, minpts=21, num_MC=200)
print(f'ZDCF bins: {len(dcf_all)}')
print(f'Lag range: {dcf_all["tau"].min():.4f} – {dcf_all["tau"].max():.1f} days')
dcf_all.head()

In [ ]:
# ── Main figure: full range + zoom ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.subplots_adjust(wspace=0.32)

ONE_DAY_COLOR = 'purple'
WEEK_COLOR    = 'green'

# LEFT: full range, log scale in days
ax_left = axes[0]
plot_zdcf(ax_left, dcf_all, label='All filters', color='steelblue',
          xscale='linear', xlim=(dcf_all['tau'].abs().min() * 0.5, dcf_all['tau'].max() * 1.5))
ax_left.axvline(1.0,    ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.6, label='1 day')
ax_left.axvline(7.0,    ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.6, label='1 week')
ax_left.axvline(30.0,   ls='--', color=WEEK_COLOR,    lw=1.0, alpha=0.5, label='1 month')
ax_left.axvline(365.25, ls='--', color='red',          lw=1.0, alpha=0.5, label='1 year')
ax_left.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax_left.set_title('Full lag range — log scale (days)', fontsize=11)
ax_left.legend(fontsize=9, loc='upper right')

# RIGHT: zoom on multi-day regime (0.5 – 100 days)
ax_right = axes[1]
dcf_zoom = dcf_all[(dcf_all['tau'] >= 0.5) & (dcf_all['tau'] <= 100.0)]
plot_zdcf(ax_right, dcf_zoom, label='All filters', color='steelblue',
          xscale='linear', xlim=(0.5, 100.0))
ax_right.axvline(1.0,  ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.6, label='1 day')
ax_right.axvline(7.0,  ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.6, label='7 days')
ax_right.axvline(30.0, ls='--', color=WEEK_COLOR,    lw=1.0, alpha=0.5, label='30 days')
ax_right.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax_right.set_title('Zoom 0.5 – 100 days', fontsize=11)
ax_right.legend(fontsize=9, loc='upper right')

fig.suptitle(
    f'PWV long-scale ZDCF (all filters) — {version_run}',
    fontsize=12, y=1.01
)
figfile1 = f'{pathfigs}/{prefix}_{version_run}_zdcf_allfilters.pdf'
fig.savefig(figfile1, bbox_inches='tight')
print(f'Saved: {figfile1}')
plt.show()

---
## 5. Zoom on intra-night regime (hours)

pyzdcf's lag axis is in days.  We convert to hours for the short-lag zoom.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12, 5))

dcf_short_hr = dcf_all.copy()
for col in ['tau', '-sig(tau)', '+sig(tau)']:
    dcf_short_hr[col] = dcf_short_hr[col] * 24.0   # days -> hours

# Keep only lags up to 12 hours
dcf_short_hr = dcf_short_hr[dcf_short_hr['tau'] <= 12.0]

plot_zdcf(ax2, dcf_short_hr, label='All filters (ZDCF)', color='steelblue',
          xscale='linear', xlim=(dcf_short_hr['tau'].abs().min() * 0.5, 12.0))
ax2.set_xlabel(r'$\Delta t$ [hours]', fontsize=13)
ax2.set_title('Intra-night regime (< 12 h) — log scale', fontsize=11)

fig2.suptitle(f'PWV ZDCF — short lags — {version_run}', fontsize=12)
figfile2 = f'{pathfigs}/{prefix}_{version_run}_zdcf_shortlags_hours.pdf'
fig2.savefig(figfile2, bbox_inches='tight')
print(f'Saved: {figfile2}')
plt.show()

---
## 6. Per-filter ZDCF — long-scale

In [ ]:
FILTERS_OF_INTEREST = ['empty', 'OG550_65mm_1', 'FELH0600']
filter_color = {
    'empty'        : '#1f77b4',
    'OG550_65mm_1' : '#d62728',
    'FELH0600'     : '#2ca02c',
}
filter_label = {
    'empty'        : 'empty',
    'OG550_65mm_1' : 'OG550',
    'FELH0600'     : 'FELH0600',
}

dcf_by_filter = {}

for filt in FILTERS_OF_INTEREST:
    sub_f = df_keep[df_keep['FILTER'] == filt].copy()
    if len(sub_f) < 30:
        print(f'  {filt}: too few points ({len(sub_f)}), skipped')
        continue

    df_dcf_f = sub_f[['t_day', 'PWV']].copy()
    df_dcf_f['sigma'] = SIGMA_REPEATABILITY

    fname_f = f'zdcf_longscale_{filt.replace(" ", "_")}.csv'
    print(f'Running pyzdcf for {filt} ({len(sub_f)} obs)...')
    dcf_f = ComputeZDCF(fname_f, df_dcf_f, minpts=11, num_MC=100)
    dcf_by_filter[filt] = dcf_f
    print(f'  -> {len(dcf_f)} bins, lag {dcf_f["tau"].min():.3f} – {dcf_f["tau"].max():.1f} days')

print('Done.')

In [ ]:
fig3, axes3 = plt.subplots(1, 2, figsize=(18, 6))
fig3.subplots_adjust(wspace=0.32)

for ax3, xlim3, xscale3, title3 in [
    (axes3[0], None,          'linear', 'Full range — log scale'),
    (axes3[1], (0.5, 100.0),  'linear', 'Zoom 0.5 – 100 days'),
]:
    # All-filter reference (grey dashed)
    x_ref = dcf_all['tau'].values
    y_ref = dcf_all['dcf'].values
    ax3.plot(x_ref, y_ref, color='black', lw=1.0, alpha=0.3, ls='--', label='All filters')

    for filt in FILTERS_OF_INTEREST:
        if filt not in dcf_by_filter:
            continue
        df_f = dcf_by_filter[filt]
        ax3.errorbar(
            df_f['tau'], df_f['dcf'],
            xerr=df_f[["-sig(tau)", "+sig(tau)"]].values.T,
            yerr=df_f[["-err(dcf)", "+err(dcf)"]].values.T,
            fmt='o', color=filter_color[filt], ms=3, lw=0.8,
            ecolor=filter_color[filt], elinewidth=1.5, capsize=2,
            label=filter_label[filt]
        )
    ax3.axhline(0.0,      ls='--', color='gray',       lw=1.0)
    ax3.axhline(1.0/np.e, ls=':',  color='darkorange', lw=1.5, label=r'$C=1/e$')
    ax3.axvline(1.0,   ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.5, label='1 day')
    ax3.axvline(7.0,   ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.5, label='7 days')
    ax3.set_xscale(xscale3)
    if xlim3 is not None:
        ax3.set_xlim(xlim3)
    ax3.set_ylim(-1.2, 1.2)
    ax3.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
    ax3.set_ylabel('ZDCF', fontsize=13)
    ax3.set_title(title3, fontsize=11)
    ax3.legend(fontsize=9, loc='upper right')

fig3.suptitle(
    f'PWV long-scale ZDCF per filter — {version_run}',
    fontsize=12, y=1.01
)
figfile3 = f'{pathfigs}/{prefix}_{version_run}_zdcf_per_filter.pdf'
fig3.savefig(figfile3, bbox_inches='tight')
print(f'Saved: {figfile3}')
plt.show()

---
## 7. ZDCF with finer binning — short to medium lags (hours to 30 days)

In [ ]:
# Run pyzdcf with smaller minpts to resolve short lags better
print('Running pyzdcf with minpts=11 (finer binning)...')
dcf_fine = ComputeZDCF('zdcf_longscale_fine.csv', df_dcf_all, minpts=11, num_MC=200)
print(f'ZDCF bins: {len(dcf_fine)}')

fig4, axes4 = plt.subplots(1, 2, figsize=(18, 6))
fig4.subplots_adjust(wspace=0.32)

# Left: lags in hours (< 24 h), log scale
ax4l = axes4[0]
dcf_fine_hr = dcf_fine.copy()
for col in ['tau', '-sig(tau)', '+sig(tau)']:
    dcf_fine_hr[col] *= 24.0
dcf_fine_hr = dcf_fine_hr[dcf_fine_hr['tau'] <= 24.0]
plot_zdcf(ax4l, dcf_fine_hr, label='All filters (minpts=11)',
          color='steelblue', xscale='log',
          xlim=(dcf_fine_hr['tau'].abs().min() * 0.5, 24.0))
ax4l.set_xlabel(r'$\Delta t$ [hours]', fontsize=13)
ax4l.set_title('Short lags (< 24 h) — log scale', fontsize=11)

# Right: lags in days (0.5–30 d), log scale
ax4r = axes4[1]
dcf_fine_day = dcf_fine[(dcf_fine['tau'] >= 0.4) & (dcf_fine['tau'] <= 30.0)]
plot_zdcf(ax4r, dcf_fine_day, label='All filters (minpts=11)',
          color='steelblue', xscale='log', xlim=(0.4, 30.0))
ax4r.axvline(1.0, ls='-',  color=ONE_DAY_COLOR, lw=1.2, alpha=0.6, label='1 day')
ax4r.axvline(7.0, ls='-',  color=WEEK_COLOR,    lw=1.2, alpha=0.6, label='7 days')
ax4r.set_xlabel(r'$\Delta t$ [days]', fontsize=13)
ax4r.set_title('Multi-day lags (0.4–30 d) — log scale', fontsize=11)
ax4r.legend(fontsize=9, loc='upper right')

fig4.suptitle(f'PWV ZDCF fine binning — {version_run}', fontsize=12, y=1.01)
figfile4 = f'{pathfigs}/{prefix}_{version_run}_zdcf_fine.pdf'
fig4.savefig(figfile4, bbox_inches='tight')
print(f'Saved: {figfile4}')
plt.show()

---
## 8. Export results

In [ ]:
csv_out = f'{pathfigs}/{prefix}_{version_run}_zdcf_allfilters.csv'
dcf_all.to_csv(csv_out, index=False, float_format='%.6f')
print(f'Saved: {csv_out}')

print(f'\n=== PWV long-scale ZDCF summary ===')
print(f'  N observations           : {len(df_keep)}')
print(f'  N nights                 : {df_keep["nightObs"].nunique()}')
print(f'  Time span                : {df_keep["t_day"].max():.1f} days')
print(f'  pyzdcf bins (minpts=21)  : {len(dcf_all)}')
print(f'  Lag range                : {dcf_all["tau"].min():.4f} – {dcf_all["tau"].max():.1f} days')